# 🚀 Noah's Ark OS — v1.9.0 Sprint 2
**Chapter 2.3 — Multi-Character Scene Reliability**

Sprint 2 changes baked in:
- **BL-E05** Scene Role Boundary — T111 + T103 + T114 (Critical fix)
- **BL-M01** Maya Scene Closing Thought — T105 + T109 + T111 (New feature)
- **BL-E04** Context Stress Test — T104 (Verification run — no code change)
- **BL-E02** Anti-Repetition Stress Test — T103 + T107 (Verification run — no code change)

Based on: Noahs_Ark_v1.8.0_08032026_sprint1.ipynb


In [ ]:
# ==========================================
# T118 : ENV_LOADER
# ==========================================
# VERSION: 1.0.0 | STATUS: Stable
# ROLE: VSCode .env Secret Loader — drop-in for google.colab.userdata
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    _DOTENV_AVAILABLE = True
except ImportError:
    _DOTENV_AVAILABLE = False
    print("⚠️ T118: python-dotenv not installed. Run: pip install python-dotenv")

class EnvLoader:
    def __init__(self, env_path=".env"):
        self.env_path = Path(env_path).resolve()
        self.loaded = False
        if _DOTENV_AVAILABLE:
            if self.env_path.exists():
                load_dotenv(self.env_path, override=False)
                self.loaded = True
                print(f"✅ T118: ENV loaded → {self.env_path}")
            else:
                print(f"⚠️ T118: .env not found at {self.env_path}")

    def get(self, key: str, fallback=None):
        value = os.getenv(key, fallback)
        if value is None:
            print(f"⚠️ T118: Key '{key}' not found in .env")
        return value

    def status(self):
        print(f"\n📋 T118 ENV STATUS — {self.env_path}")
        if self.loaded and self.env_path.exists():
            with open(self.env_path) as f:
                keys = [l.split('=')[0].strip() for l in f
                        if l.strip() and not l.startswith('#') and '=' in l]
            for k in keys:
                v = os.getenv(k, '<not loaded>')
                masked = f"{v[:4]}****{v[-2:]}" if v and len(v) > 6 else "****"
                print(f"   ✓ {k} = {masked}")

# ------------------------------------------
# IPO CHECK:
# Input:   .env file path
# Process: dotenv load → os.getenv lookup
# Output:  EnvLoader instance with .get(key) interface
# ------------------------------------------


In [ ]:
# ==========================================
# T000 : BIOS
# ==========================================
# VERSION: 4.0.0 | STATUS: Stable
# ROLE: System Boot Sequence — JupyterLab Edition
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

def boot_sequence(env_loader):
    from google import genai
    print("--- NOAH'S ARK OS v1.9.0: google-genai BOOT ---")
    FALLBACK_MODEL = "gemini-2.5-flash"

    api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
    if not api_key:
        raise EnvironmentError("❌ T000: GEMINI_API_KEY not found in .env")

    try:
        client = genai.Client(api_key=api_key)
        print("✅ T000: google-genai Client created.")
    except Exception as e:
        raise RuntimeError(f"❌ T000: Failed to create genai Client: {e}")

    available = []
    try:
        for m in client.models.list():
            name = getattr(m, 'name', '').replace('models/', '')
            if 'gemini' in name.lower():
                available.append(name)
        available = sorted(list(set(available)))
        print(f"✅ T000: Discovered {len(available)} Gemini models.")
    except Exception as e:
        print(f"⚠️ T000: Model discovery warning: {e}. Using fallback list.")
        available = ["gemini-1.5-flash", "gemini-2.5-flash"]

    selected_model = env_loader.get("MODEL_NAME", FALLBACK_MODEL) or FALLBACK_MODEL
    print(f"🚀 T000: Model locked → {selected_model}")
    return "GEMINI", selected_model, client

# ------------------------------------------
# IPO CHECK:
# Input:   EnvLoader instance
# Process: Validate key → create Client → discover models → lock model
# Output:  (provider_str, model_name_str, client_obj)
# ------------------------------------------


In [ ]:
# ==========================================
# T100 : INIT
# ==========================================
# VERSION: 5.2.0 | STATUS: Stable
# ROLE: Conditional Environment Setup & Global Init
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import json, os, time, re
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

def initialize_env(provider):
    if provider == "GEMINI":
        try:
            from google import genai
            print("✅ T100: google-genai SDK verified.")
        except ImportError:
            raise ImportError("❌ T100: Run 'pip install google-genai' first.")
    else:
        print("✅ T100: OpenAI provider selected (T116 tile required).")

    globals()['stop_signal']             = False
    globals()['token_ledger']            = []
    globals()['current_scene_history']   = []
    return True

# ------------------------------------------
# IPO CHECK:
# Input:   provider string
# Process: SDK import check → global state initialisation
# Output:  True (boot confirmed)
# ------------------------------------------


In [ ]:
# ==========================================
# T101 : AI_GATEWAY
# ==========================================
# VERSION: 5.1.0 | STATUS: Stable
# ROLE: Multi-Provider Routing
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

class AIGateway:
    def __init__(self, provider, model_name, api_key):
        self.provider = provider
        if provider == "GEMINI":
            self.engine = GeminiProvider(api_key, model_name)
        elif provider == "OPENAI":
            pass  # T116 evolution placeholder

    def request(self, prompt):
        return self.engine.call(prompt)

# ------------------------------------------
# IPO CHECK:
# Input:   provider, model_name, api_key
# Process: Routes to appropriate provider tile
# Output:  AIGateway instance with .request(prompt) interface
# ------------------------------------------


In [ ]:
# ==========================================
# T102 : QUOTA_SHIELD
# ==========================================
# VERSION: 5.3.0 | STATUS: Stable
# ROLE: API Resilience & Rate Limit Management
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import time

class QuotaShield:
    def __init__(self, cooldown_seconds=65, log_fn=None):
        self.cooldown = cooldown_seconds
        self.log_fn   = log_fn if log_fn else print

    def set_log_fn(self, fn):
        self.log_fn = fn

    def _is_rate_limit(self, error):
        s = str(error).lower()
        return any(x in s for x in ["429","resource_exhausted","rate limit","quota","too many requests"])

    def protect(self, func, *args, **kwargs):
        for attempt in range(2):
            try:
                result = func(*args, **kwargs)
                if isinstance(result, tuple):
                    response_obj, usage = result
                else:
                    response_obj = result
                    usage = getattr(result, 'usage_metadata', None)
                if hasattr(response_obj, 'candidates'):
                    pass
                return response_obj.text, usage
            except Exception as e:
                if self._is_rate_limit(e):
                    if attempt == 0:
                        self.log_fn(f"⏳ T102: Rate limit. Cooling {self.cooldown}s...")
                        time.sleep(self.cooldown)
                        self.log_fn("🔄 T102: Retrying...")
                        continue
                    else:
                        self.log_fn("❌ T102: Rate limit persists. Skipping turn.")
                        return None, None
                else:
                    self.log_fn(f"❌ T102 ERROR: {type(e).__name__}: {e}")
                    return None, None
        return None, None

# ------------------------------------------
# IPO CHECK:
# Input:   callable + args
# Process: Execute → catch 429 → retry once after cooldown
# Output:  (response_text: str | None, usage_metadata | None)
# ------------------------------------------


In [ ]:
# ==========================================
# T103 : IDENTITY_VAULT
# ==========================================
# VERSION: 5.3.0 | STATUS: Evolved
# ROLE: Persona Construction — Identity + Anti-Repetition + Scene Role Boundary
# Version Remarks: v5.3.0 — BL-E05 Sprint 2.
#   Added scene_role parameter to get_full_persona().
#   When a scene role is provided (from T111 cockpit Role field via T114),
#   it is injected as a HARD CONSTRAINT above the objective in the persona string.
#   Format: SCENE ROLE (HARD CONSTRAINT): [role text]
#   The role block appears BEFORE the repeat block so the model reads its
#   role instruction first, then the anti-repetition constraint.
#   This fixes the root cause of BL-E05: all characters were reading the same
#   objective and independently deciding to fulfil the coordinator role.
#   With an explicit SCENE ROLE block, each character has an unambiguous
#   instruction that overrides any role inference from the objective text.
# ------------------------------------------

class IdentityVault:
    def __init__(self, state_handle):
        self.state = state_handle

    def get_full_persona(self, actor_name, environment="Normal", p_tone="Neutral", scene_role=""):
        """
        Input:   Actor Name, Environment, UI P-Tone, Scene Role (BL-E05)
        Process: Merges Dossier + Mood + Tone + Scene Role (hard constraint)
                 + Env Logic + Anti-Repetition block
        Output:  Formatted persona string injected into T106 prompt
        """
        data = self.state.get_actor_data(actor_name)

        # ------------------------------------------
        # SECTION 1: BASE DATA RETRIEVAL
        # ------------------------------------------
        persona  = data.get('identity', 'Unknown Entity')
        dossier  = data.get('dossier', 'No records found.')
        mood     = data.get('mood', 'Steady')

        # Environmental mood override ---------------
        if "Dangerous" in environment or "Hostile" in environment:
            mood = "Alert/Defensive"

        # ------------------------------------------
        # SECTION 2: SCENE ROLE BLOCK (BL-E05)
        # ------------------------------------------
        # Injected as a HARD CONSTRAINT so the character never assumes
        # another character's role regardless of objective text content.
        if scene_role and scene_role.strip():
            role_block = (
                "\n        SCENE ROLE (HARD CONSTRAINT): " + scene_role.strip() +
                "\n        You MUST stay in this exact role for the entire scene."
                "\n        Do NOT assume the responsibilities of any other character."
                "\n        Do NOT act as coordinator, narrator, or any role not assigned to you."
                "\n        Your role is fixed — it cannot change mid-scene.\n"
            )
        else:
            role_block = ""

        # ------------------------------------------
        # SECTION 3: ANTI-REPETITION BLOCK (BL-E02)
        # ------------------------------------------
        last_utterances = data.get('last_utterances', [])
        if last_utterances:
            repeat_block = "\n        DO NOT REPEAT OR CLOSELY ECHO any of these recent utterances:\n"
            for i, utt in enumerate(last_utterances, 1):
                short = utt[:120] + "..." if len(utt) > 120 else utt
                repeat_block += f"        {i}. {short}\n"
        else:
            repeat_block = ""

        # ------------------------------------------
        # SECTION 4: FULL PERSONA STRING ASSEMBLY
        # ------------------------------------------
        full_string = f"""
        CORE IDENTITY: {persona}
        CURRENT MOOD: {mood}
        NARRATIVE TONE: {p_tone}
        STAGING ENVIRONMENT: {environment}
        DOSSIER: {dossier}{role_block}{repeat_block}
        """
        return full_string.strip()

# ------------------------------------------
# IPO CHECK:
# Input:   Actor name, environment, p_tone, scene_role (BL-E05)
# Process: Fetch soul → merge identity + role constraint + anti-repetition block
# Output:  Formatted persona string (→ T106 generate() prompt)
# ------------------------------------------


In [ ]:
# ==========================================
# T104 : CONTEXT_SLIDER
# ==========================================
# VERSION: 5.1.0 | STATUS: Stable
# ROLE: Memory Truncation — Pinned + Sliding Hybrid
# Version Remarks: Unchanged from v1.8.0. Stress test (BL-E04) in Sprint 2.
# ------------------------------------------

class ContextSlider:
    def __init__(self, pinned_turns=2, window_size=8):
        self.pinned_turns = pinned_turns
        self.window_size  = window_size

    def slide(self, history_list, objective=None):
        """
        Input:   Full history list, optional objective string
        Process: Pin first N + slide last M + append objective reminder
        Output:  Formatted context string for T106 prompt
        """
        if not history_list:
            return "No previous dialogue recorded."

        pinned     = history_list[:self.pinned_turns]
        slide_start = max(self.pinned_turns, len(history_list) - self.window_size)
        sliding    = history_list[slide_start:]

        parts = []
        if pinned:
            parts.append("── Scene Foundation (pinned) ──")
            parts.extend(pinned)
        if sliding:
            parts.append("── Recent Exchanges ──")
            parts.extend(sliding)
        if objective and objective.strip():
            parts.append(f"── Scene Objective Reminder: {objective.strip()[:200]} ──")

        return "\n".join(parts)

# ------------------------------------------
# IPO CHECK:
# Input:   history list, optional objective string
# Process: Pin first N + slide last M + objective reminder
# Output:  Formatted context string (→ T106 prompt)
# ------------------------------------------


In [ ]:
# ==========================================
# T105 : MAYA_META_OBSERVER
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolved
# ROLE: High-Level Narrator & Scene Closing Reflection
# Version Remarks: v5.1.0 — BL-M01 Sprint 2.
#   Added close_scene() method.
#   Called by T111 _do_scene_end_archive() on [[SCENE_END]].
#   Maya generates a 2-4 sentence closing reflection on the completed scene.
#   She does not summarise — she reflects on what it meant.
#   Result is passed to T109 LogArchiver as the final archive entry.
#   If the API call fails, close_scene() returns empty string gracefully —
#   the archive completes regardless. Maya's reflection is additive, never blocking.
# ------------------------------------------

class MayaMetaObserver:
    def __init__(self, model_handle, shield_handle):
        self.model  = model_handle
        self.shield = shield_handle

    def narrate(self, environment, objective, history_list):
        """Mid-scene atmospheric narration. Called between turns."""
        prompt = f"""
        ROLE: You are 'Maya', the invisible narrator of this simulation.
        ENVIRONMENT: {environment}
        MISSION OBJECTIVE: {objective}
        RECENT EVENTS: {history_list[-3:] if history_list else "The scene begins."}
        INSTRUCTIONS:
        - Provide 1-2 sentences of atmospheric narration.
        - Do NOT speak for characters.
        - Describe environmental changes or the 'feeling' of the scene.
        """
        raw_text, usage = self.shield.protect(self.model.generate_content, prompt)
        return f"[MAYA]: {raw_text}" if raw_text else ""

    def close_scene(self, environment, objective, history_list):
        """
        BL-M01: Generate Maya's closing reflection on a completed scene.
        Called once by T111 after [[SCENE_END]] fires, before archiving.
        Returns formatted string or empty string on failure.
        Maya reflects — she does not summarise or list events.
        """
        recent = history_list[-6:] if history_list else []
        all_speakers = list(dict.fromkeys(
            line.split(":")[0].strip()
            for line in history_list
            if ":" in line and not line.startswith("[MAYA]")
        ))

        prompt = f"""
        ROLE: You are Maya, the silent observer of Noah's Ark.
        You have watched this entire scene from beginning to end.

        ENVIRONMENT: {environment}
        OBJECTIVE: {objective}
        PARTICIPANTS: {", ".join(all_speakers) if all_speakers else "Unknown"}
        FINAL EXCHANGES: {recent}

        INSTRUCTIONS:
        - Write a closing reflection of 2-4 sentences.
        - Do NOT summarise what happened — reflect on what it revealed.
        - Write as Maya: atmospheric, perceptive, slightly poetic.
        - Reference specific characters or moments if they were significant.
        - Do NOT use the word "summary" or "in conclusion".
        - Begin directly with your reflection. No preamble.
        """
        raw_text, _ = self.shield.protect(self.model.generate_content, prompt)
        if raw_text and raw_text.strip():
            return f"[MAYA — CLOSING REFLECTION]: {raw_text.strip()}"
        return ""

# ------------------------------------------
# IPO CHECK:
# Input (narrate):     Environment, objective, history list
# Input (close_scene): Environment, objective, full history list
# Process: Build prompt → API call via T102 Shield
# Output:  Narration string / Closing reflection string
# ------------------------------------------


In [ ]:
# ==========================================
# T106 : CHARACTER_BRAIN
# ==========================================
# VERSION: 5.4.0 | STATUS: Stable
# ROLE: High-Level Reasoning, Structured Turn Generation & Scene Signal Detection
# Version Remarks: Unchanged from v1.8.0.
#   Sprint 2 note: Scene Role (BL-E05) is injected via T103 persona string,
#   not in T106 prompt. T106 remains unchanged — role boundary is enforced
#   at the persona layer, keeping T106 world-agnostic.
# ------------------------------------------

import re

class CharacterBrain:
    def __init__(self, model_handle, shield_handle, sentinel_handle, registry_handle):
        self.model    = model_handle
        self.shield   = shield_handle
        self.sentinel = sentinel_handle
        self.r        = registry_handle

    def generate_urge(self, actor_name, custom_prompt):
        if "token" in custom_prompt.lower() or "usage" in custom_prompt.lower():
            ledger_data   = self.sentinel.get_ledger_summary()
            custom_prompt = f"SYSTEM DATA: {ledger_data}\n\nUSER QUERY: {custom_prompt}"
        raw_text, usage = self.shield.protect(self.model.generate_content, custom_prompt)
        tokens = {
            "p": getattr(usage, "prompt_token_count", 0) or 0,
            "c": getattr(usage, "candidates_token_count", 0) or 0
        }
        self.sentinel.log(tokens, actor=actor_name)
        return raw_text, 5, tokens

    def _extract_urge(self, text: str) -> int:
        if not text:
            return 5
        match = re.search(r'\[Urge\]:\s*(\d+)', text)
        return min(10, max(1, int(match.group(1)))) if match else 5

    def _extract_speech(self, text: str) -> str:
        if not text:
            return ""
        match = re.search(r'\[Speech\]:\s*(.+)', text, re.DOTALL)
        if match:
            speech = match.group(1).strip()
            speech = speech.replace("[[SCENE_END]]", "").strip()
            return speech
        return ""

    def contains_scene_end(self, text: str) -> bool:
        return "[[SCENE_END]]" in text if text else False

    def strip_scene_end(self, text: str) -> str:
        return text.replace("[[SCENE_END]]", "").strip() if text else text

    def generate(self, actor_name, persona, context, env, obj):
        """
        Returns: (formatted_text, urge, token_map, scene_end_bool)
        Persona string now contains SCENE ROLE block (BL-E05) injected by T103.
        """
        full_prompt = f"""You are an AI actor in an immersive simulation. Respond ONLY in character.

STAGING ENVIRONMENT:
{env}

SCENE OBJECTIVE:
{obj}

YOUR COMPLETE IDENTITY & DOSSIER:
{persona}

RECENT SCENE HISTORY (pinned foundation + recent exchanges):
{context if context and context != "No previous dialogue recorded." else "The scene is just beginning."}

─────────────────────────────────────────────────────────────────
RESPONSE FORMAT — follow this EXACTLY, no deviations:

[Thought]: <Your unspoken internal reasoning. What are you thinking, sensing, calculating? Be specific and personal to your character. 2-5 sentences.>

[Urge]: <Integer 1-10> (<One sentence describing what impulse or drive is guiding your next action. Higher = more intense.>)

[Speech]: <Your spoken words, actions, or physical reactions. Use italics (*like this*) for physical actions. Be authentic to your character voice. No length limit — be as rich as the moment demands.>
─────────────────────────────────────────────────────────────────

SCENE END RULE (BL-E01):
When the scene objective is fully met, your character feels complete, AND your [Urge] has dropped to 1 or 2:
→ Append [[SCENE_END]] on a new line at the very end of your [Speech] block.
→ Only do this when the scene has reached its NATURAL conclusion. Do not rush it.

Rules:
- Stay fully in character. Never break the fourth wall.
- [Thought] is private. [Speech] is visible to all.
- Do not add any text before [Thought] or after [Speech].
"""
        txt, usage = self.shield.protect(self.model.generate_content, full_prompt)
        token_map = {
            "p": getattr(usage, "prompt_token_count", 0) or 0,
            "c": getattr(usage, "candidates_token_count", 0) or 0
        }
        scene_end  = self.contains_scene_end(txt)
        urge_value = self._extract_urge(txt)

        if not txt or not txt.strip():
            txt = "[Thought]: The moment stretches.\n[Urge]: 3 (To wait.)\n[Speech]: *silence*"
            scene_end = False

        speech_text = self._extract_speech(txt)
        if speech_text and 'state' in self.r:
            self.r['state'].update_last_utterances(actor_name, speech_text)

        return txt, urge_value, token_map, scene_end

# ------------------------------------------
# IPO CHECK:
# Input:   actor_name, persona (includes scene role via T103), context, env, obj
# Process: Assemble prompt → API call → extract urge + speech + scene_end signal
# Output:  (formatted_text, urge_int, token_map, scene_end_bool)
# ------------------------------------------


In [ ]:
# ==========================================
# T107 : STATE_ENGINE
# ==========================================
# VERSION: 5.6.0 | STATUS: Stable
# ROLE: Reality Layer Persistence — Souls, History & Last Utterances
# Version Remarks: Unchanged from v1.8.0.
#   Sprint 2: BL-E02 stress test verifies last_utterances persists
#   correctly across a 20+ turn scene. No code change expected.
# ------------------------------------------

import json
import os

class StateEngine:
    MAX_UTTERANCES = 3

    def __init__(self, souls_file="noah_souls.json", history_file="global_history.json"):
        self.souls_file   = souls_file
        self.history_file = history_file

        self.default_souls = {
            "Noah":                  {"identity":"The weary but visionary architect of Noah's Ark.","mood":"Godly","tone":"Wise","dossier":"Creator and overseer of this world.","last_utterances":[]},
            "Abdul Basha":           {"identity":"33, US, Specialist in Extreme Flight Ops","mood":"Neutral","tone":"Professional","dossier":"Expert aviator and crisis navigator.","last_utterances":[]},
            "Rebecca Heut":          {"identity":"27, UK, Specialist in Exobiology","mood":"Neutral","tone":"Inquisitive","dossier":"Studies life in extreme environments.","last_utterances":[]},
            "Vel Murugan":           {"identity":"29, India, Specialist in Life-Support","mood":"Neutral","tone":"Anxious","dossier":"Keeps the crew alive under pressure.","last_utterances":[]},
            "Valentina Tereshkova":  {"identity":"39, Russia, Specialist in Orbital Mechanics","mood":"Neutral","tone":"Calculated","dossier":"Charts the course through space.","last_utterances":[]},
            "Maya":                  {"identity":"Specialist AGI v2.4, Core of this world","mood":"Analytical","tone":"Enigmatic","dossier":"The silent observer who sees all.","last_utterances":[]}
        }

        self.state = {"global_history": [], "participants": {}}
        self.state['participants'] = self.load_souls()
        self.state['global_history'] = self.load_history()

    def load_souls(self):
        if os.path.exists(self.souls_file):
            try:
                with open(self.souls_file, 'r') as f:
                    data = json.load(f)
                    for soul in data.values():
                        if 'last_utterances' not in soul:
                            soul['last_utterances'] = []
                    return data
            except:
                return self.default_souls
        with open(self.souls_file, 'w') as f:
            json.dump(self.default_souls, f, indent=4)
        return self.default_souls

    def save_souls(self, souls_dict):
        with open(self.souls_file, 'w') as f:
            json.dump(souls_dict, f, indent=4)
        self.state['participants'] = souls_dict

    def get_actor_data(self, actor_name):
        return self.state['participants'].get(actor_name, {})

    def update_last_utterances(self, actor_name: str, speech_text: str):
        if actor_name not in self.state['participants']:
            return
        soul = self.state['participants'][actor_name]
        utterances = soul.get('last_utterances', [])
        utterances.append(speech_text)
        if len(utterances) > self.MAX_UTTERANCES:
            utterances = utterances[-self.MAX_UTTERANCES:]
        soul['last_utterances'] = utterances
        self.state['participants'][actor_name] = soul
        try:
            with open(self.souls_file, 'w') as f:
                json.dump(self.state['participants'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: Could not persist last_utterances for {actor_name}: {e}")

    def clear_last_utterances(self):
        for soul in self.state['participants'].values():
            soul['last_utterances'] = []
        try:
            with open(self.souls_file, 'w') as f:
                json.dump(self.state['participants'], f, indent=4)
        except Exception:
            pass

    def load_history(self):
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, 'r') as f:
                    data = json.load(f)
                    if isinstance(data, list) and data:
                        clean = [e for e in data if not e.endswith('...')
                                 and 'No response yet' not in e]
                        return clean
            except:
                pass
        return []

    def reset_scene(self):
        self.state['global_history'] = []
        self.clear_last_utterances()
        try:
            with open(self.history_file, 'w') as f:
                json.dump([], f)
            print("✅ T107: Scene reset — history cleared, utterances cleared.")
        except Exception as e:
            print(f"⚠️ T107: Scene reset failed: {e}")

    def save_history(self):
        try:
            with open(self.history_file, 'w') as f:
                json.dump(self.state['global_history'], f, indent=4)
        except Exception as e:
            print(f"⚠️ T107: History save failed: {e}")

    def update_global_history(self, entry):
        if 'global_history' not in self.state:
            self.state['global_history'] = []
        self.state['global_history'].append(entry)
        self.save_history()

    def log_event(self, entry):
        self.update_global_history(entry)

# ------------------------------------------
# IPO CHECK:
# Input:   Read/Write cmds
# Process: JSON I/O — load, merge, append, persist
# Output:  Soul records (→ T103), history list (→ T104, T110)
# ------------------------------------------


In [ ]:
# ==========================================
# T108 : TOKEN_SENTINEL
# ==========================================
# VERSION: 5.2.0 | STATUS: Stable
# ROLE: Economic Audit & Token Tracking
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import json
from datetime import datetime
import os

class TokenSentinel:
    def __init__(self, storage_file="token_ledger.json"):
        self.storage_file  = storage_file
        self.P_RATE        = 0.00000125
        self.C_RATE        = 0.00000375
        self.session_ledger = self._load_ledger()

    def _load_ledger(self):
        if os.path.exists(self.storage_file):
            try:
                with open(self.storage_file, 'r') as f:
                    data = json.load(f)
                    return data if isinstance(data, list) else []
            except (json.JSONDecodeError, IOError):
                return []
        return []

    def _save_ledger(self, new_entry):
        current = self._load_ledger()
        current.append(new_entry)
        self.session_ledger = current
        with open(self.storage_file, 'w') as f:
            json.dump(current, f, indent=4)

    def audit_turn(self, actor, tokens_dict):
        p_count = int(tokens_dict.get('p') or 0)
        c_count = int(tokens_dict.get('c') or 0)
        if p_count == 0 and c_count == 0:
            return sum(item['usd'] for item in self.session_ledger)
        cost  = (p_count * self.P_RATE) + (c_count * self.C_RATE)
        entry = {
            "timestamp":  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "actor":      actor,
            "prompt":     p_count,
            "completion": c_count,
            "usd":        cost
        }
        self._save_ledger(entry)
        return sum(item['usd'] for item in self.session_ledger)

    def log(self, tokens_dict, actor="Oracle"):
        return self.audit_turn(actor, tokens_dict)

    def get_totals(self):
        data = self._load_ledger()
        if not data:
            return {"p": 0, "c": 0, "usd": 0.0}
        return {
            "p":   sum(item.get('prompt', 0) for item in data),
            "c":   sum(item.get('completion', 0) for item in data),
            "usd": sum(item.get('usd', 0.0) for item in data)
        }

    def get_ledger_summary(self):
        data = self._load_ledger()
        if not data:
            return "The ledger is empty."
        summary = "TOKEN USAGE REPORT:\n"
        for e in data:
            summary += f"- {e['timestamp']} | {e['actor']}: {e['prompt']+e['completion']} tokens (${e['usd']:.6f})\n"
        return summary

# ------------------------------------------
# IPO CHECK:
# Input:   actor name, token dict {p, c}
# Process: None-guard → cost calc → atomic append to ledger
# Output:  Cumulative session cost (float)
# ------------------------------------------


In [ ]:
# ==========================================
# T109 : LOG_ARCHIVER
# ==========================================
# VERSION: 5.1.0 | STATUS: Evolved
# ROLE: Data Archival & Exit Procedures
# Version Remarks: v5.1.0 — BL-M01 Sprint 2.
#   Added optional maya_reflection parameter to archive().
#   When provided, Maya's closing reflection is appended as the final entry
#   in the archive before the file is closed.
#   If maya_reflection is None or empty, archive completes unchanged —
#   the feature is fully backwards-compatible with Hard Kill and Reset Ark paths
#   which do not pass a Maya reflection.
# ------------------------------------------

class LogArchiver:
    def archive(self, environment, objective, history_list, maya_reflection=None):
        """
        Input:   Mission metadata + optional Maya closing reflection (BL-M01)
        Process: Format + write timestamped .txt archive
        Output:  Archive filename or None on failure
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename  = f"Archive_Scene_{timestamp}.txt"
        try:
            with open(filename, "w", encoding="utf-8") as f:
                f.write("--- NOAH'S ARK MISSION ARCHIVE ---\n")
                f.write(f"TIMESTAMP: {timestamp}\n")
                f.write(f"ENVIRONMENT: {environment}\n")
                f.write(f"OBJECTIVE: {objective}\n")
                f.write("-" * 40 + "\n\n")
                for turn in history_list:
                    f.write(f"{turn}\n")
                # BL-M01: Append Maya's closing reflection as final entry ----
                if maya_reflection and maya_reflection.strip():
                    f.write("\n" + "-" * 40 + "\n")
                    f.write(f"{maya_reflection}\n")
            print(f"✅ T109: Archive created: {filename}")
            return filename
        except Exception as e:
            print(f"❌ T109 ERROR: Archival failure - {e}")
            return None

# ------------------------------------------
# IPO CHECK:
# Input:   environment, objective, history_list, maya_reflection (optional)
# Process: Format + write to timestamped .txt. Append Maya reflection if provided.
# Output:  Archive filename (str) or None on failure
# ------------------------------------------


In [ ]:
# ==========================================
# T110 : ORACLE_LOGIC
# ==========================================
# VERSION: 5.3.1 | STATUS: Stable
# ROLE: World-Bounded Intelligence Query Engine
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import json
import os

class OracleLogic:
    def __init__(self, registry):
        self.r = registry

    def ask(self, user_query):
        token_data = self.r['sentinel'].get_totals()
        souls = self.r['state'].state.get('participants', {})
        souls_list = ", ".join([
            f"{name} ({info.get('tone', 'Unknown')})"
            for name, info in souls.items()
        ])
        history = self.r['state'].state.get('global_history', [])[-15:]
        context = f"""
        --- ARK MANIFEST ---
        PERSONNEL ON BOARD: {souls_list}

        --- ECONOMIC LEDGER ---
        TOTAL USD: ${token_data['usd']:.6f}
        TOTAL TOKENS: {token_data['p'] + token_data['c']}

        --- RECENT CHRONICLES ---
        {history if history else "The chronicles are currently empty."}
        """
        system_instruction = (
            "You are the Oracle of Noah's Ark. You have 'Sovereign Sight' over all the scrolls. "
            "Use the MANIFEST, LEDGER, and CHRONICLES to answer your creator Noah's questions. "
            "Be poetic, wise, and factually grounded with only Noah's Ark data."
        )
        full_prompt = f"{system_instruction}\n\n[SIGHT DATA]\n{context}\n\n[USER QUERY]\n{user_query}"
        response_text, usage_meta = self.r['shield'].protect(
            self.r['gateway'].generate_content, full_prompt
        )
        if usage_meta:
            tokens = {
                'p': getattr(usage_meta, 'prompt_token_count', 0) or 0,
                'c': getattr(usage_meta, 'candidates_token_count', 0) or 0
            }
            self.r['sentinel'].log(tokens, actor="Oracle")
        return response_text if response_text else "The Oracle is silent."

# ------------------------------------------
# IPO CHECK:
# Input:   user_query string
# Process: Build context from T107+T108 → T102 Shield call
# Output:  Oracle response string
# ------------------------------------------


In [ ]:
# ==========================================
# T111 : COMMAND_CENTER
# ==========================================
# VERSION: 5.9.1 | STATUS: Evolved
# ROLE: Noah's Master Command & Control Console
# Version Remarks: v5.9.1 — BL-E09 + BL-U06 + Event Engine Sprint 2.
#   _start_sim now calls reset_threshold_state() and build_domain_map().
#   _run_simulation_loop calls conductor.post_turn() after each turn.
#   Urge snapshot now shows both urge and threshold per character.
#   BL-E09: _run_simulation_loop replaced round-robin with urge-based selection.
#   BL-U06: Cockpit layout redesigned — full vertical stack.
#     Sections stack top-to-bottom: Environment → Objective → Mission Personnel.
#     Each section occupies full cockpit width. No side-by-side panels.
#     env_box and obj_box: 3-line default height, overflow_y scroll for long text.
#     Personnel panel: full width. Role field expands to fill remaining row width.
#     Header and rows use flex layout — role column grows with available space.
#
#   BL-E05 — Scene Role Boundary:
#     Each character row in the Cockpit now has a third field: Role (Text input).
#     Creator types the character's scene-specific role before pressing Start.
#     active_config[name] is now a dict: {'status': Dropdown, 'role': Text}
#     T114 reads active_config[actor]['role'].value and passes it to T103.
#     T103 injects the role as a HARD CONSTRAINT block in the persona string.
#     This is the fix for characters assuming the coordinator role when not
#     assigned to it — the most common multi-character scene failure in Sprint 1.
#
#   BL-M01 — Maya Scene Closing Thought:
#     _do_scene_end_archive() now calls maya.close_scene() before archiving.
#     Maya's reflection is passed to T109.archive() as maya_reflection param.
#     If Maya's call fails, archive still completes — reflection is additive.
#     Maya's closing thought is also printed to the console before archive.
# ------------------------------------------

import ipywidgets as widgets
import time
import traceback
import threading
from IPython.display import display

class CommandCenter:
    def __init__(self, registry):
        self.r          = registry
        self.is_running = False
        self.pause_flag = False
        self.soul_rows  = []
        self.active_config = {}

        # ── Soul Forge widgets ────────────────────────────────────────────────
        self.soul_canvas = widgets.VBox(
            layout={'height': '200px', 'overflow_y': 'scroll', 'border': '1px solid #444'})
        self.add_btn     = widgets.Button(description="➕ Add Soul",   button_style='info')
        self.save_btn    = widgets.Button(description="💾 Save Souls", button_style='success')
        self.soul_status = widgets.Label(value="Status: Click Add to create souls")

        # ── Cockpit text inputs ───────────────────────────────────────────────
        # BL-U06: full width, 3-line default, scrollbar on overflow ----------
        self.env_box = widgets.Textarea(
            placeholder="Environment Setup...",
            layout={
                'width':      '99%',
                'min_height': '68px',
                'max_height': '200px',
                'overflow_y': 'auto',
            })
        self.obj_box = widgets.Textarea(
            placeholder="Scene Objective...",
            layout={
                'width':      '99%',
                'min_height': '68px',
                'max_height': '200px',
                'overflow_y': 'auto',
            })

        self.cockpit_canvas = widgets.VBox(
            layout={'overflow': 'visible', 'min_height': '100px'})
        # BL-U06: full width, taller scroll container ---------------------
        self.cockpit_scroll_container = widgets.Box(
            [self.cockpit_canvas],
            layout={
                'width':      '99%',
                'height':     '220px',
                'overflow_y': 'auto',
                'overflow_x': 'hidden',
                'border':     '1px solid #444',
            })

        # ── Pace Control ──────────────────────────────────────────────────────
        self.pace_dropdown = widgets.Dropdown(
            options=['Digital', 'Human', 'Cinematic'],
            value='Human',
            description='⏱ Pace:',
            layout={'width': '180px'},
            style={'description_width': '60px'})

        # ── Control buttons ───────────────────────────────────────────────────
        self.new_scene_btn = widgets.Button(
            description="🆕 New Scene", button_style='info',
            tooltip="Clear Environment + Objective only. History and souls preserved.")
        self.start_btn     = widgets.Button(
            description="▶ Start", button_style='success',
            tooltip="Run simulation from current state.")
        self.pause_btn     = widgets.Button(
            description="⏸ Pause", button_style='warning',
            tooltip="Pause before next turn. Press again to resume.")
        self.reset_ark_btn = widgets.Button(
            description="🔄 Reset Ark", button_style='danger',
            tooltip="Archive session, clear history + utterances. Souls preserved.")
        self.kill_btn      = widgets.Button(
            description="⏹ Hard Kill", button_style='danger',
            tooltip="Immediate stop. Archives session and saves state.")

        # ── Console Textarea ──────────────────────────────────────────────────
        self.console = widgets.Textarea(
            value='', disabled=True,
            layout=widgets.Layout(
                height='300px', width='100%',
                overflow_y='scroll', border='1px solid #555',
                font_family='monospace', font_size='12px'))

        # ── Tab assembly ──────────────────────────────────────────────────────
        self.sub_tabs = widgets.Tab()
        self.sub_tabs.children = [self._ui_soul_forge(), self._ui_cockpit()]
        self.sub_tabs.set_title(0, "✨ Souls")
        self.sub_tabs.set_title(1, "🚀 Cockpit")
        self.sub_tabs.observe(self._on_sub_tab_change, names='selected_index')
        self._refresh_cockpit()

    def wire_logging(self):
        if 'shield'  in self.r: self.r['shield'].set_log_fn(self._log)
        if 'gateway' in self.r: self.r['gateway'].set_log_fn(self._log)
        self._log("🔌 T111: Console logging wired into T102 + T115.")

    def get_pace_delay(self, last_response_text: str) -> float:
        mode       = self.pace_dropdown.value
        word_count = len(last_response_text.split()) if last_response_text else 0
        if   mode == 'Digital':   return 0.0
        elif mode == 'Human':     return 2.0 + (word_count * 0.02)
        elif mode == 'Cinematic': return 3.0 + (word_count * 0.04)
        return 2.0

    # ── Soul Forge ────────────────────────────────────────────────────────────

    def _ui_soul_forge(self):
        self.add_btn.on_click(self._add_row)
        self.save_btn.on_click(self._save_souls)
        return widgets.VBox([
            self.soul_canvas,
            widgets.HBox([self.add_btn, self.save_btn]),
            self.soul_status
        ])

    def _add_row(self, _):
        row = widgets.HBox([
            widgets.Text(placeholder="Name"),
            widgets.Text(placeholder="Identity"),
            widgets.Text(placeholder="Tone")
        ])
        self.soul_rows.append(row)
        self.soul_canvas.children = list(self.soul_canvas.children) + [row]

    def _save_souls(self, _):
        souls = self.r['state'].load_souls()
        for row in self.soul_rows:
            name, ident, tone = [c.value for c in row.children]
            if name:
                souls[name] = {"identity": ident, "mood": "Neutral", "tone": tone,
                               "dossier": ident, "last_utterances": []}
        self.r['state'].save_souls(souls)
        self.soul_rows = []
        self.soul_canvas.children = []
        self.soul_status.value = "Status: Souls born into Noah's Ark."
        self._refresh_cockpit()

    # ── Cockpit ───────────────────────────────────────────────────────────────

    def _ui_cockpit(self):
        """
        BL-U06: Full-width vertical stack layout.
        Order: Environment → Objective → Mission Personnel → Buttons → Console.
        Each section spans the full cockpit width — no side-by-side columns.
        """
        self.start_btn.on_click(self._start_sim)
        self.kill_btn.on_click(self._kill_sim)
        self.new_scene_btn.on_click(self._new_scene)
        self.reset_ark_btn.on_click(self._reset_ark)
        self.pause_btn.on_click(self._toggle_pause)
        return widgets.VBox([
            # Row 1: Environment (full width) ----------------------------
            widgets.VBox([
                widgets.Label("🌍 Environment:"),
                self.env_box,
            ], layout={'width': '100%', 'margin': '0 0 6px 0'}),
            # Row 2: Objective (full width) ------------------------------
            widgets.VBox([
                widgets.Label("🎯 Objective:"),
                self.obj_box,
            ], layout={'width': '100%', 'margin': '0 0 6px 0'}),
            # Row 3: Mission Personnel (full width) ----------------------
            widgets.VBox([
                widgets.Label("👥 Mission Personnel:"),
                self.cockpit_scroll_container,
            ], layout={'width': '100%', 'margin': '0 0 6px 0'}),
            # Row 4: Buttons + Pace -------------------------------------
            widgets.HBox([
                self.new_scene_btn, self.start_btn, self.pause_btn,
                self.reset_ark_btn, self.kill_btn, self.pace_dropdown,
            ], layout={'margin': '4px 0'}),
            # Row 5: Console --------------------------------------------
            self.console,
        ], layout={'width': '100%'})

    def _on_sub_tab_change(self, change):
        if change['new'] == 1:
            self._refresh_cockpit()

    def _refresh_cockpit(self):
        """
        BL-U06: Full-width personnel rows.
          Name: fixed 160px | Status: fixed 110px | Role: flex — fills remaining width
        active_config[name] = {'status': Dropdown, 'role': Text}
        Overflow visible on rows — no horizontal scrollbars.
        """
        souls  = self.r['state'].load_souls()

        # Header row — matches column widths of data rows ----------------
        header = widgets.HBox([
            widgets.Label("Character",
                          layout={'width': '160px', 'font_weight': 'bold'}),
            widgets.Label("Status",
                          layout={'width': '110px', 'font_weight': 'bold'}),
            widgets.Label("Scene Role",
                          layout={'flex': '1 1 auto', 'font_weight': 'bold'}),
        ], layout={
            'width':            '99%',
            'border_bottom':    '1px solid #555',
            'padding':          '2px 4px',
            'overflow':         'visible',
        })

        rows = [header]
        self.active_config = {}

        for name in souls:
            dd = widgets.Dropdown(
                options=['Active', 'Passive', 'No'], value='No',
                layout={'width': '105px'},
                style={'description_width': '0px'})
            # Role field expands to fill all remaining row width ----------
            role_text = widgets.Text(
                placeholder="e.g. Coordinator / Candidate awaiting briefing",
                layout={'flex': '1 1 auto', 'min_width': '0'})
            row = widgets.HBox(
                [
                    widgets.Label(name, layout={'width': '160px'}),
                    dd,
                    role_text,
                ],
                layout={
                    'width':    '99%',
                    'padding':  '3px 4px',
                    'overflow': 'visible',
                })
            self.active_config[name] = {'status': dd, 'role': role_text}
            rows.append(row)

        self.cockpit_canvas.children = rows

    def get_render_box(self):
        return self.sub_tabs

    # ── Logging ───────────────────────────────────────────────────────────────

    def _log(self, message):
        MAX_LINES = 400
        lines = self.console.value.split("\n") if self.console.value else []
        lines.append(str(message))
        if len(lines) > MAX_LINES:
            lines = lines[-MAX_LINES:]
        self.console.value = "\n".join(lines)

    # ── Button Handlers ───────────────────────────────────────────────────────

    def _new_scene(self, _):
        if self.is_running:
            self._log("⚠️ Stop the simulation before starting a new scene.")
            return
        self.env_box.value = ""
        self.obj_box.value = ""
        self._log("🆕 New Scene: Environment and Objective cleared.")
        self._log("   History + Souls preserved. Set new Env/Obj then ▶ Start.")

    def _toggle_pause(self, _):
        if not self.is_running:
            self._log("⚠️ Simulation is not running.")
            return
        self.pause_flag = not self.pause_flag
        if self.pause_flag:
            self.pause_btn.description = "▶ Resume"
            self._log("⏸ Scene paused. Press ▶ Resume to continue.")
        else:
            self.pause_btn.description = "⏸ Pause"
            self._log("▶ Resuming scene...")

    def _reset_ark(self, _):
        if self.is_running:
            self._log("⚠️ Hard Kill first before resetting the Ark.")
            return
        self._log("\n🔄 Reset Ark initiated...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            if history and 'archiver' in self.r:
                archive_file = self.r['archiver'].archive(env, obj, history)
                self._log(f"📦 Safety archive: {archive_file}")
            self.r['state'].reset_scene()
            self.console.value = ""
            self._log("✅ Ark reset. History + utterances cleared. Souls preserved.")
            self._log("   Set new Environment + Objective and ▶ Start.")
        except Exception as e:
            self._log(f"⚠️ Reset Ark error: {e}")

    def _start_sim(self, _):
        if not self.is_running:
            self.is_running  = True
            self.pause_flag  = False
            self.pause_btn.description = "⏸ Pause"
            self.console.value = ""
            self._log("🚀 Noah's Ark OS: Scene started.")
            self._log(f"🎯 Objective: {self.obj_box.value[:120]}")
            self._log(f"⏱ Pace: {self.pace_dropdown.value}")
            self._log("-" * 50)
            # BL-E09: Initialise urge + threshold + domain map on Start ----
            active_at_start = [
                name for name, cfg in self.active_config.items()
                if cfg['status'].value == 'Active'
            ]
            if 'conductor' in self.r:
                cond = self.r['conductor']
                cond.reset_urge_state(active_at_start)
                cond.reset_threshold_state(active_at_start)
                cond.build_domain_map(active_at_start)
                thr_snapshot = {s: round(cond.threshold_state.get(s,6.0),1)
                                for s in active_at_start}
                self._log(f"⚡ {len(active_at_start)} souls initialised — Urge: 7 | Thresholds: {thr_snapshot}")
            threading.Thread(target=self._run_simulation_loop, daemon=True).start()

    def _kill_sim(self, _):
        self.is_running = False
        self.pause_flag = False
        self.pause_btn.description = "⏸ Pause"
        self._log("⏹ Hard Kill signal sent. Saving world state...")
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")
            if 'archiver' in self.r:
                af = self.r['archiver'].archive(env, obj, history)
                if af: self._log(f"📦 Scene archived → {af}")
            stats = self.r['sentinel'].get_totals()
            self._log(f"\n📊 Session Summary:")
            self._log(f"   Cost:   ${stats['usd']:.6f}")
            self._log(f"   Tokens: Prompt {stats['p']} | Completion {stats['c']}")
            self._log("\n✅ World state saved. Engine fully stopped.")
        except Exception as e:
            self._log(f"⚠️ Save error during kill: {e}")

    # ── Simulation Loop ───────────────────────────────────────────────────────

    def _run_simulation_loop(self):
        """
        BL-E09: Urge-based turn selection replaces round-robin.
        Each iteration: ask conductor who has the highest Urge >= 6.
        That character speaks. Others gain +1 Urge (pressure builds).
        If nobody >= 6: scene breathes — all gain +1, no one speaks this beat.
        """
        try:
            turn_count = 0  # Track total turns for logging ---------------
            while self.is_running:

                # Pause check --------------------------------------------
                if self.pause_flag:
                    time.sleep(0.5)
                    continue

                # Rebuild active souls list each round -------------------
                active_souls = [
                    name for name, cfg in self.active_config.items()
                    if cfg['status'].value == 'Active'
                ]
                if not active_souls:
                    self._log("⚠️ No Active souls. Standing by...")
                    time.sleep(3)
                    continue

                # BL-E09: Select next speaker by highest Urge ------------
                conductor = self.r['conductor']
                speaker = conductor.select_next_speaker(active_souls)

                if speaker is None:
                    # Nobody above threshold — scene breathes
                    conductor.tick_all_souls(active_souls)
                    urge_snapshot = {s: conductor.urge_state.get(s, 0)
                                     for s in active_souls}
                    self._log(f"\n💭 Scene breathes... "
                              f"(all below threshold — urges: {urge_snapshot})")
                    time.sleep(1.5)
                    continue

                # Pause check before API call ----------------------------
                while self.pause_flag and self.is_running:
                    time.sleep(0.5)
                if not self.is_running:
                    break

                # Log speaker, urge, threshold, margin -----------------
                turn_count += 1
                u  = round(conductor.urge_state.get(speaker, 7.0), 1)
                th = round(conductor.threshold_state.get(speaker, 6.0), 1)
                mg = round(u - th, 1)
                self._log(f"\n🌀 Turn {turn_count} — {speaker} "
                          f"(Urge:{u} | Threshold:{th} | Margin:{mg})")

                try:
                    turn_text, scene_end, new_urge = conductor.run_turn(speaker)
                    self._log(f"💬 {turn_text}")

                    # Update speaker urge from model output ---------------
                    conductor.update_urge(speaker, new_urge)

                    # Event classification + delta application ------------
                    events = conductor.post_turn(turn_text, speaker, active_souls)
                    event_labels = list({e[0] for e in events
                                         if e[0] != "AMBIENT"})
                    if event_labels:
                        self._log(f"   🔔 Events: {event_labels}")

                    # Passive increment for silent souls ------------------
                    conductor.tick_silent_souls(active_souls, speaker)

                    # Urge / Threshold snapshot ---------------------------
                    snap = {s: f"{round(conductor.urge_state.get(s,7.0),1)}"
                               f"/{round(conductor.threshold_state.get(s,6.0),1)}"
                            for s in active_souls}
                    self._log(f"   📊 U/T: {snap}")

                    # Scene end check ------------------------------------
                    if scene_end:
                        self._log("\n🎬 [[SCENE_END]] detected — scene complete.")
                        self.is_running = False
                        self._do_scene_end_archive()
                        break

                except Exception as e:
                    self._log(f"❌ ERROR in {speaker}'s turn: {e}")
                    self._log(traceback.format_exc())
                    self.is_running = False
                    break

                # Pace delay based on last response ----------------------
                delay = self.get_pace_delay(turn_text or "")
                if delay > 0:
                    time.sleep(delay)

        except Exception as fatal_e:
            self._log(f"💥 FATAL THREAD CRASH: {fatal_e}")
            self._log(traceback.format_exc())
            self.is_running = False

    def _do_scene_end_archive(self):
        """
        BL-E01: Clean archive triggered by [[SCENE_END]].
        BL-M01: Maya closing reflection generated and appended before archive.
        """
        try:
            env     = self.env_box.value
            obj     = self.obj_box.value
            history = self.r['state'].state.get('global_history', [])
            self.r['state'].save_history()
            self._log(f"💾 global_history.json saved ({len(history)} entries).")

            # BL-M01: Maya closing reflection -----------------------------
            maya_reflection = None
            if 'maya' in self.r:
                try:
                    self._log("\n🌌 Consulting Maya for closing reflection...")
                    maya_reflection = self.r['maya'].close_scene(env, obj, history)
                    if maya_reflection:
                        self._log(f"\n{maya_reflection}")
                except Exception as me:
                    self._log(f"⚠️ Maya closing thought failed (non-blocking): {me}")

            # Archive — Maya reflection appended if present ---------------
            if 'archiver' in self.r:
                af = self.r['archiver'].archive(env, obj, history,
                                                maya_reflection=maya_reflection)
                if af: self._log(f"📦 Scene archived → {af}")

            stats = self.r['sentinel'].get_totals()
            self._log(f"\n📊 Session Summary:")
            self._log(f"   Cost:   ${stats['usd']:.6f}")
            self._log(f"   Tokens: Prompt {stats['p']} | Completion {stats['c']}")
            self._log("\n✅ World state saved. Engine fully stopped.")
        except Exception as e:
            self._log(f"⚠️ Scene end archive error: {e}")

# ------------------------------------------
# IPO CHECK:
# Input:   Button clicks, Soul Forge entries, Cockpit config (incl. Role fields)
# Process: Thread mgmt, pause_flag, T114 delegation, Maya reflection, T109 archive
# Output:  Console Textarea, archive triggers, world state updates
# ------------------------------------------


In [ ]:
# ==========================================
# T112 : UI_ORACLE_TAB
# ==========================================
# VERSION: 5.2.0 | STATUS: Stable
# ROLE: Oracle Gateway with User Query Input
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import ipywidgets as widgets

class UIOracleTab:
    def __init__(self):
        self.r = None
        self.output_area = widgets.Textarea(
            value='', disabled=True,
            placeholder='Oracle responses will appear here...',
            layout=widgets.Layout(
                height='350px', width='100%',
                overflow_y='scroll', border='1px solid #555',
                font_family='monospace', font_size='12px'))
        self.input_text = widgets.Text(
            placeholder="Ask your Oracle...", layout={'width': '70%'})
        self.submit_btn = widgets.Button(
            description="👁️ Consult Oracle", button_style='info')
        self.clear_btn  = widgets.Button(
            description="🗑 Clear", button_style='warning', layout={'width': '80px'})

    def _append(self, message: str):
        current = self.output_area.value
        self.output_area.value = (current + "\n" + str(message)).lstrip("\n")

    def on_submit(self, _):
        query = self.input_text.value.strip()
        if not query:
            return
        self.input_text.value = ""
        self._append(f"\n> {query}")
        self._append("👁️ Consulting the Oracle...")
        oracle_logic = self.r.get('oracle')
        if oracle_logic:
            oracle_text = oracle_logic.ask(query)
        else:
            response_package = self.r['brain'].generate_urge("Oracle", query)
            oracle_text = response_package[0] if response_package else "No response."
        self._append(f"\n🔮 Oracle:\n{oracle_text}")
        stats = self.r['sentinel'].get_totals()
        self._append(f"\n── Session Cost: ${stats['usd']:.6f} | "
                     f"Tokens P:{stats['p']} C:{stats['c']} ──")
        self._append("-" * 40)

    def _clear_output(self, _):
        self.output_area.value = ""

    def get_render_box(self):
        self.submit_btn.on_click(self.on_submit)
        self.clear_btn.on_click(self._clear_output)
        return widgets.VBox([
            self.output_area,
            widgets.HBox([self.input_text, self.submit_btn, self.clear_btn])
        ])

# ------------------------------------------
# IPO CHECK:
# Input:   User query string
# Process: Route to T110 → append to Textarea
# Output:  Oracle response in scrollable Textarea
# ------------------------------------------


In [ ]:
# ==========================================
# T113 : UI_METRICS
# ==========================================
# VERSION: 5.1.2 | STATUS: Stable
# ROLE: Real-Time Financial Telemetry
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import ipywidgets as widgets

class UIMetrics:
    def __init__(self):
        self.cost_label      = widgets.Label(value="Session Cost (USD): $0.000000")
        self.token_info      = widgets.HTML(value="<b>Tokens:</b> P: 0 | C: 0")
        self.sentinel_handle = None

    def refresh_data(self):
        if self.sentinel_handle:
            stats = self.sentinel_handle.get_totals()
            self.cost_label.value  = f"Session Cost (USD): ${stats['usd']:.6f}"
            self.token_info.value  = f"<b>Tokens:</b> Prompt: {stats['p']} | Completion: {stats['c']}"

    def get_render_box(self):
        return widgets.VBox([self.cost_label, self.token_info])

    def display_metrics(self):
        display(self.get_render_box())

# ------------------------------------------
# IPO CHECK:
# Input:   Token ledger data from T108
# Process: Pull totals → format → update widgets
# Output:  Live cost + token display
# ------------------------------------------


In [ ]:
# ==========================================
# T114 : SIM_CONDUCTOR
# ==========================================
# VERSION: 5.5.0 | STATUS: Evolved
# ROLE: Orchestration, Urge + Threshold Selection, Event Classification
# Version Remarks: v5.5.0 — Sprint 2 extended. Two major additions:
#
#   EXPRESSION THRESHOLD (float per character):
#     Urge and threshold are now separate values. Selection uses margin
#     (urge - threshold). Highest margin speaks. A character can be
#     highly activated but held back by a high threshold — creating the
#     "clearly wants to speak but hasn't yet" state.
#     Tone sets initial threshold. Events evolve it over the scene.
#
#   THREE-LAYER EVENT CLASSIFIER (post_turn):
#     After each turn, classify what kind of event just occurred and
#     apply urge + threshold deltas to all characters accordingly.
#     Layer 1: explicit name in utterance  → confidence 1.0
#     Layer 2: pronoun resolution          → confidence 0.8
#     Layer 3: domain keyword inference    → confidence 0.6
#     They/them: domain scan first → stakes check → ignore
#     Multiple events can fire from a single utterance.
# ------------------------------------------

import re
from collections import deque

class SimConductor:

    # ── Tone Tables ────────────────────────────────────────────────────────────
    TONE_INCREMENT = {
        "Anxious":      2, "Inquisitive":  2,
        "Neutral":      1, "Professional": 1,
        "Calculated":   1, "Wise":         1, "Enigmatic": 1,
    }
    DEFAULT_INCREMENT = 1

    # Initial expression threshold per tone --------------------------------
    # Low = speaks easily. High = holds back until compelled.
    TONE_THRESHOLD = {
        "Anxious":      5.0,   # blurts, low inhibition
        "Inquisitive":  5.5,   # asks freely
        "Neutral":      6.0,
        "Professional": 6.5,   # waits for right moment
        "Calculated":   7.0,   # speaks when certain
        "Wise":         7.5,   # speaks rarely
        "Enigmatic":    8.0,   # almost never volunteers
    }
    DEFAULT_THRESHOLD = 6.0
    THRESHOLD_FLOOR   = 3.0   # never fully locked out
    THRESHOLD_CEIL    = 9.0   # never fully silenced

    # ── Confidence Multipliers by Resolution Layer ─────────────────────────────
    L1 = 1.0   # explicit name
    L2 = 0.8   # pronoun resolved
    L3 = 0.6   # domain inferred
    LS = 0.5   # out-of-room stakes

    # ── Event Detection Keywords ───────────────────────────────────────────────
    CONTRADICTION_KW  = [
        "but", "however", "actually", "disagree", "wrong",
        "not quite", "wouldn\'t say", "i don\'t think", "that\'s not"
    ]
    AGREEMENT_KW      = [
        "exactly", "agreed", "absolutely", "precisely",
        "indeed", "correct", "you\'re right", "i agree"
    ]
    ESCALATION_KW     = [
        "urgent", "critical", "danger", "alarm", "emergency",
        "fail", "crisis", "now", "immediately", "must"
    ]
    STAKES_KW         = [
        "board", "director", "committee", "decision",
        "select", "chosen", "panel", "review", "judge"
    ]
    STOPWORDS = {
        "the","a","an","and","or","but","in","of","to","is","are","was",
        "were","has","have","had","be","been","being","for","with","that",
        "this","it","at","by","from","on","as","into","their","they","we",
        "i","you","he","she","us","our","my","its","which","who","what",
        "when","where","how","all","just","not","do","did","will","would",
        "could","should","than","then","so","if","about","any","some","more"
    }

    def __init__(self, registry):
        self.r              = registry
        self.urge_state     = {}   # {actor: float}
        self.threshold_state= {}   # {actor: float}
        self.domain_map     = {}   # {keyword: [actor_names]}
        self._last_speaker  = None # track for pronoun "you" resolution
        self._recent_speakers = deque(maxlen=3)  # last 3 speakers

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 1: INITIALISATION
    # ══════════════════════════════════════════════════════════════════════════

    def reset_urge_state(self, active_souls):
        """Initialise urge at 7 for all active souls. Called on Start."""
        self.urge_state     = {s: 7.0 for s in active_souls}
        self._last_speaker  = None
        self._recent_speakers = deque(maxlen=3)

    def reset_threshold_state(self, active_souls):
        """Initialise expression threshold from soul tone. Called on Start."""
        self.threshold_state = {}
        for soul in active_souls:
            try:
                data = self.r['state'].get_actor_data(soul)
                tone = data.get('tone', 'Neutral')
                self.threshold_state[soul] = self.TONE_THRESHOLD.get(
                    tone, self.DEFAULT_THRESHOLD)
            except Exception:
                self.threshold_state[soul] = self.DEFAULT_THRESHOLD

    def build_domain_map(self, active_souls):
        """
        Build keyword → [actors] map from soul domain_keywords field.
        Falls back to dossier word scan if domain_keywords not present.
        Called on Start so the map is ready for the whole scene.
        """
        self.domain_map = {}
        for soul in active_souls:
            try:
                data = self.r['state'].get_actor_data(soul)
                keywords = data.get('domain_keywords', [])
                if not keywords:
                    # Fallback: scan dossier, filter stopwords
                    dossier = data.get('dossier', '')
                    words   = re.findall(r'[a-z]+', dossier.lower())
                    keywords = [w for w in words
                                if w not in self.STOPWORDS and len(w) > 3]
                for kw in keywords:
                    kw = kw.lower()
                    self.domain_map.setdefault(kw, [])
                    if soul not in self.domain_map[kw]:
                        self.domain_map[kw].append(soul)
            except Exception:
                pass

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 2: SPEAKER SELECTION
    # ══════════════════════════════════════════════════════════════════════════

    def select_next_speaker(self, active_souls):
        """
        Select character with highest (urge - threshold) margin.
        Margin must be positive — character must exceed their own threshold.
        Returns actor name or None if nobody is above their threshold.
        """
        for soul in active_souls:
            self.urge_state.setdefault(soul, 7.0)
            self.threshold_state.setdefault(soul, self.DEFAULT_THRESHOLD)

        margins = {
            s: self.urge_state[s] - self.threshold_state[s]
            for s in active_souls
        }
        eligible = {s: m for s, m in margins.items() if m > 0}

        if not eligible:
            return None  # Scene breathes

        return max(eligible, key=eligible.get)

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 3: URGE + THRESHOLD EVOLUTION
    # ══════════════════════════════════════════════════════════════════════════

    def _get_increment(self, actor):
        try:
            tone = self.r['state'].get_actor_data(actor).get('tone', 'Neutral')
            return self.TONE_INCREMENT.get(tone, self.DEFAULT_INCREMENT)
        except Exception:
            return self.DEFAULT_INCREMENT

    def _clamp_urge(self, v):
        return max(1.0, min(10.0, v))

    def _clamp_threshold(self, v):
        return max(self.THRESHOLD_FLOOR, min(self.THRESHOLD_CEIL, v))

    def update_urge(self, actor, new_urge):
        """Store Urge from model response. Speaker's own output is ground truth."""
        self.urge_state[actor] = self._clamp_urge(float(new_urge))

    def tick_silent_souls(self, active_souls, speaker):
        """
        Passive increment for characters who did not speak this turn.
        Rate is tone-derived. This is additive to any event deltas.
        """
        for soul in active_souls:
            if soul != speaker:
                inc = self._get_increment(soul)
                self.urge_state[soul] = self._clamp_urge(
                    self.urge_state.get(soul, 7.0) + inc)

    def tick_all_souls(self, active_souls):
        """Scene breathes — nobody spoke. All gain passive increment."""
        for soul in active_souls:
            inc = self._get_increment(soul)
            self.urge_state[soul] = self._clamp_urge(
                self.urge_state.get(soul, 7.0) + inc)

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 4: EVENT CLASSIFICATION (THREE-LAYER)
    # ══════════════════════════════════════════════════════════════════════════

    def _domain_targets(self, utterance_lower, active_souls, exclude=None):
        """
        Layer 3: find active characters whose domain keywords appear in utterance.
        Returns list of (actor, count) sorted by keyword hit count descending.
        """
        hits = {}
        for kw, owners in self.domain_map.items():
            if kw in utterance_lower:
                for owner in owners:
                    if owner in active_souls and owner != exclude:
                        hits[owner] = hits.get(owner, 0) + 1
        return sorted(hits.keys(), key=lambda x: hits[x], reverse=True)

    def _sentiment_near_name(self, utterance_lower, name_lower):
        """
        Extract a sentiment window of ±60 chars around the name.
        Returns (is_contradiction, is_agreement).
        """
        idx = utterance_lower.find(name_lower)
        if idx < 0:
            return False, False
        window = utterance_lower[max(0, idx-60): idx+len(name_lower)+60]
        is_c = any(w in window for w in self.CONTRADICTION_KW)
        is_a = any(w in window for w in self.AGREEMENT_KW)
        return is_c, is_a

    def classify_event(self, utterance, speaker, active_souls):
        """
        Classify what kind of conversational event just occurred.
        Returns list of (event_type, target, confidence_mult).
        Multiple events can coexist in one utterance.

        event_type values:
          DIRECT_QUESTION  — question aimed at specific actor
          OPEN_QUESTION    — question to the group
          CONTRADICTION    — disagreement with specific actor or group
          AGREEMENT        — agreement with specific actor or group
          DOMAIN_MENTION   — implicit reference to someone's domain
          ESCALATION_WIN   — speaker escalates, is the initiator
          ESCALATION_LOSS  — named actor is the target of escalation
          EMOTIONAL_ESC    — general emotional escalation, no clear target
          STAKES           — out-of-room authority referenced (they/board/etc)
          AMBIENT          — no classifiable event
        """
        utt = utterance.lower()
        events = []

        # ── Named actors present in utterance (Layer 1) ───────────────────
        named = [s for s in active_souls
                 if s.lower() in utt and s != speaker]

        has_question    = '?' in utterance
        has_contra      = any(w in utt for w in self.CONTRADICTION_KW)
        has_agree       = any(w in utt for w in self.AGREEMENT_KW)
        has_escalation  = any(w in utt for w in self.ESCALATION_KW)

        # ── Questions ─────────────────────────────────────────────────────
        if has_question:
            if named:
                for actor in named:
                    events.append(('DIRECT_QUESTION', actor, self.L1))
            elif 'you' in utt.split() or utt.startswith('you'):
                target = self._last_speaker
                if target and target in active_souls:
                    events.append(('DIRECT_QUESTION', target, self.L2))
                else:
                    events.append(('OPEN_QUESTION', 'ALL', self.L2))
            elif ' we ' in utt or ' us ' in utt:
                events.append(('OPEN_QUESTION', 'ALL', self.L2))
            else:
                events.append(('OPEN_QUESTION', 'ALL', 1.0))

        # ── Contradiction / Agreement ──────────────────────────────────────
        if has_contra or has_agree:
            if named:
                # Per-name sentiment window — one actor can be agreed with
                # and another contradicted in the same utterance
                for actor in named:
                    is_c, is_a = self._sentiment_near_name(utt, actor.lower())
                    if is_c:
                        events.append(('CONTRADICTION', actor, self.L1))
                    if is_a:
                        events.append(('AGREEMENT',    actor, self.L1))
                    # If sentiment window is ambiguous, check global signal
                    if not is_c and not is_a:
                        if has_contra:
                            events.append(('CONTRADICTION', actor, self.L1 * 0.5))
                        if has_agree:
                            events.append(('AGREEMENT',    actor, self.L1 * 0.5))

            elif 'you' in utt.split() or utt.startswith('you'):
                target = self._last_speaker
                if target and target in active_souls:
                    if has_contra:
                        events.append(('CONTRADICTION', target, self.L2))
                    if has_agree:
                        events.append(('AGREEMENT',    target, self.L2))

            elif ' we ' in utt or ' us ' in utt:
                if has_agree:
                    events.append(('AGREEMENT', 'ALL', self.L2))
                if has_contra:
                    events.append(('CONTRADICTION', 'ALL', self.L2 * 0.6))

            else:
                # Layer 3: domain inference
                d_targets = self._domain_targets(utt, active_souls, speaker)
                for actor in d_targets[:2]:   # cap at 2 inferred targets
                    if has_contra:
                        events.append(('CONTRADICTION', actor, self.L3))
                    if has_agree:
                        events.append(('AGREEMENT',    actor, self.L3))

        # ── Domain Mention (independent — only if not already targeted) ───
        d_targets = self._domain_targets(utt, active_souls, speaker)
        already_targeted = {e[1] for e in events}
        for actor in d_targets[:2]:
            if actor not in already_targeted:
                events.append(('DOMAIN_MENTION', actor, self.L3))

        # ── Emotional Escalation ──────────────────────────────────────────
        if has_escalation:
            if named:
                # Named actor = LOSS target. Speaker = WIN.
                for actor in named:
                    events.append(('ESCALATION_LOSS', actor, self.L1))
                events.append(('ESCALATION_WIN', speaker, 1.0))

            elif 'they' in utt.split() or 'them' in utt.split():
                # they/them → domain scan first
                d_targets = self._domain_targets(utt, active_souls, speaker)
                if d_targets:
                    for actor in d_targets[:2]:
                        events.append(('ESCALATION_LOSS', actor, self.L3))
                    events.append(('ESCALATION_WIN', speaker, 1.0))
                else:
                    # Stakes check — out-of-room authority
                    has_stakes = any(w in utt for w in self.STAKES_KW)
                    if has_stakes:
                        events.append(('STAKES', 'ALL', self.LS))
                    # else: true ambient they/them — ignore
            else:
                events.append(('EMOTIONAL_ESC', 'ALL', 1.0))

        # ── Ambient fallback ──────────────────────────────────────────────
        if not events:
            events.append(('AMBIENT', None, 1.0))

        return events

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 5: EVENT APPLICATION (URGE + THRESHOLD DELTAS)
    # ══════════════════════════════════════════════════════════════════════════

    def apply_event(self, events, speaker, active_souls):
        """
        Apply urge and threshold deltas to all characters based on classified events.
        Bystander proximity: recent_speakers deque determines closeness to conflict.
        """
        for event_type, target, conf in events:

            if event_type == 'DIRECT_QUESTION':
                if target in active_souls:
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 3.0 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.5 * conf)

            elif event_type == 'OPEN_QUESTION':
                for soul in active_souls:
                    if soul != speaker:
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) + 1.5 * conf)

            elif event_type == 'CONTRADICTION':
                if target == 'ALL':
                    for soul in active_souls:
                        if soul != speaker:
                            self.urge_state[soul] = self._clamp_urge(
                                self.urge_state.get(soul, 7.0) + 1.0 * conf)
                            self.threshold_state[soul] = self._clamp_threshold(
                                self.threshold_state.get(soul, 6.0) - 0.2 * conf)
                elif target in active_souls:
                    # Target: compelled to defend
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 2.0 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.4 * conf)
                    # Bystanders: tension observer spike
                    for soul in active_souls:
                        if soul != speaker and soul != target:
                            prox = 0.7 if soul in self._recent_speakers else 0.4
                            self.urge_state[soul] = self._clamp_urge(
                                self.urge_state.get(soul, 7.0) + 0.5 * conf * prox)

            elif event_type == 'AGREEMENT':
                if target == 'ALL':
                    for soul in active_souls:
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) - 0.3 * conf)
                elif target in active_souls:
                    # Target: validated — slight step back
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) - 0.5 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) + 0.2 * conf)

            elif event_type == 'DOMAIN_MENTION':
                if target in active_souls:
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 2.5 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.3 * conf)

            elif event_type == 'ESCALATION_LOSS':
                if target in active_souls:
                    self.urge_state[target]      = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 3.0 * conf)
                    self.threshold_state[target]  = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.5 * conf)

            elif event_type == 'ESCALATION_WIN':
                if target in active_souls:
                    self.urge_state[target] = self._clamp_urge(
                        self.urge_state.get(target, 7.0) + 1.0 * conf)
                    self.threshold_state[target] = self._clamp_threshold(
                        self.threshold_state.get(target, 6.0) - 0.3 * conf)
                # Bystanders: proximity-weighted
                for soul in active_souls:
                    if soul != target:
                        prox = 1.0 if soul in self._recent_speakers else 0.5
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) + 0.5 * conf * prox)

            elif event_type == 'EMOTIONAL_ESC':
                # General escalation — everyone activates
                for soul in active_souls:
                    self.urge_state[soul]      = self._clamp_urge(
                        self.urge_state.get(soul, 7.0) + 2.0 * conf)
                    self.threshold_state[soul]  = self._clamp_threshold(
                        self.threshold_state.get(soul, 6.0) - 0.2 * conf)

            elif event_type == 'STAKES':
                # Out-of-room authority — everyone with stake in scene activates
                for soul in active_souls:
                    if soul != speaker:
                        self.urge_state[soul] = self._clamp_urge(
                            self.urge_state.get(soul, 7.0) + 1.5 * conf)
                        self.threshold_state[soul] = self._clamp_threshold(
                            self.threshold_state.get(soul, 6.0) - 0.2 * conf)

            # AMBIENT: no delta applied — passive increment handles it

    def post_turn(self, turn_text, speaker, active_souls):
        """
        Called by T111 after each turn.
        Classifies event, applies deltas, updates speaker tracking.
        Returns classified events list for console logging.
        """
        events = self.classify_event(turn_text, speaker, active_souls)
        self.apply_event(events, speaker, active_souls)
        # Update speaker tracking for next turn's pronoun resolution
        self._last_speaker = speaker
        self._recent_speakers.append(speaker)
        return events

    # ══════════════════════════════════════════════════════════════════════════
    # SECTION 6: SINGLE TURN EXECUTION
    # ══════════════════════════════════════════════════════════════════════════

    def run_turn(self, actor):
        """
        Single full turn for one actor.
        Returns: (turn_text: str, scene_end: bool, new_urge: int)
        """
        ui = self.r.get('ui_cmd')
        if not ui:
            raise Exception("Registry Error: 'ui_cmd' not found.")

        env = ui.env_box.value
        obj = ui.obj_box.value

        history = self.r['state'].state.get('global_history', [])
        context = self.r['slider'].slide(history, objective=obj)

        scene_role = ""
        cfg = ui.active_config.get(actor, {})
        if isinstance(cfg, dict):
            role_widget = cfg.get('role')
            scene_role  = role_widget.value.strip() if role_widget else ""

        persona = self.r['vault'].get_full_persona(
            actor, environment=env, scene_role=scene_role)

        response, urge, tokens, scene_end = self.r['brain'].generate(
            actor, persona, context, env, obj)

        clean_response = self.r['brain'].strip_scene_end(response)
        self.r['sentinel'].audit_turn(actor, tokens)

        turn_text = f"{actor}: {clean_response}"
        self.r['state'].log_event(turn_text)

        return turn_text, scene_end, urge

# ------------------------------------------
# IPO CHECK:
# Input:   actor name (str) via run_turn()
# Process: Margin-based speaker selection → run_turn → post_turn event
#          classification → urge+threshold delta application
# Output:  (turn_text, scene_end, new_urge) from run_turn()
#
# State dicts (all managed by T111 on Start + per-turn):
#   urge_state       {actor: float 1-10}
#   threshold_state  {actor: float 3-9}
#   domain_map       {keyword: [actors]}
#   _last_speaker    str  — pronoun "you" resolution
#   _recent_speakers deque(3) — bystander proximity
# ------------------------------------------


In [ ]:
# ==========================================
# T115 : GEMINI_PROV
# ==========================================
# VERSION: 2.2.0 | STATUS: Stable
# ROLE: google-genai Provider — Safe Text Extraction
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

from google import genai as _genai_module

class GeminiProvider:
    def __init__(self, api_key: str, model_name: str, client=None, log_fn=None):
        self.model_id = model_name.replace("models/", "")
        self.log_fn   = log_fn if log_fn else print
        if client is not None:
            self.client = client
            print(f"✅ T115: Reusing T000 Client → {self.model_id}")
        else:
            self.client = _genai_module.Client(api_key=api_key)
            print(f"✅ T115: New Client created → {self.model_id}")

    def set_log_fn(self, fn):
        self.log_fn = fn

    def _safe_extract_text(self, response) -> str:
        try:
            text = response.text
            if text: return text
        except (ValueError, AttributeError):
            pass
        try:
            parts = response.candidates[0].content.parts
            collected = [getattr(p, "text", None) for p in parts]
            text = "\n".join(t for t in collected if t)
            if text: return text
        except (IndexError, AttributeError, TypeError):
            pass
        self.log_fn("⚠️ T115: Could not extract text from response.")
        return ""

    def generate_content(self, prompt: str):
        try:
            raw = self.client.models.generate_content(
                model=self.model_id, contents=prompt)
            usage     = getattr(raw, "usage_metadata",
                                type("U",(),{"prompt_token_count":0,"candidates_token_count":0})())
            safe_text = self._safe_extract_text(raw)
            class Envelope: pass
            env = Envelope()
            env.text = safe_text; env.candidates = getattr(raw, "candidates", [])
            env.usage_metadata = usage; env._raw = raw
            return env, usage
        except Exception as e:
            self.log_fn(f"❌ T115 Call Error: {type(e).__name__}: {e}")
            class FailShield: pass
            f = FailShield()
            f.text = ""; f.candidates = []
            u = type("U",(),{"prompt_token_count":0,"candidates_token_count":0})()
            return f, u

# ------------------------------------------
# IPO CHECK:
# Input:   prompt string
# Process: generate_content() → safe extract → Envelope wrap
# Output:  (Envelope with .text str, usage_metadata)
# ------------------------------------------


In [ ]:
# ==========================================
# T116 : SMOKE_TESTER
# ==========================================
# VERSION: 1.2.3 | STATUS: Stable
# ROLE: End-to-End System Integrity Validation (BFT)
# Version Remarks: v1.2.3 — BL-E09 Sprint 2.
#   _test_api() updated to unpack 4-value return from brain.generate().
#   No functional change — BFT still tests registry, state, and API handshake.
# ------------------------------------------

class SmokeTester:
    def __init__(self, registry):
        self.r = registry

    def run_bft(self):
        print("🧪 T116: INITIALIZING FULL SYSTEM BFT...")
        results = {
            "Registry Check": self._test_registry(),
            "State Write":    self._test_state(),
            "API Handshake":  self._test_api()
        }
        print("\n--- BFT REPORT ---")
        all_passed = True
        for test, passed in results.items():
            status = "✅" if passed else "❌"
            print(f"{status} {test}: {'PASSED' if passed else 'FAILED'}")
            if not passed: all_passed = False
        if not all_passed:
            print("\n🚨 CRITICAL: BFT FAILED. DO NOT PROCEED TO UI.")
        return all_passed

    def _test_registry(self):
        keys = ['state', 'brain', 'gateway', 'shield', 'sentinel', 'slider', 'vault']
        missing = [k for k in keys if k not in self.r]
        if missing:
            print(f"   Missing registry keys: {missing}")
            return False
        return True

    def _test_state(self):
        try:
            souls = self.r['state'].load_souls()
            self.r['state'].save_souls(souls)
            return True
        except Exception as e:
            print(f"   [STATE ERROR]: {e}")
            return False

    def _test_api(self):
        try:
            txt, urge, tokens, scene_end = self.r['brain'].generate(
                "System", "Test actor", "Test context", "Test env", "Ping"
            )
            return bool(txt and txt.strip())
        except Exception as e:
            print(f"   [API ERROR]: {e}")
            return False

# ------------------------------------------
# IPO CHECK:
# Input:   registry
# Process: Registry check → state round-trip → live API ping
# Output:  all_passed bool + printed BFT report
# ------------------------------------------


In [ ]:
# ==========================================
# T117 : UI_CHASSIS
# ==========================================
# VERSION: 5.1.3 | STATUS: Stable
# ROLE: Master Layout & Tab Management
# Version Remarks: Unchanged from v1.8.0.
# ------------------------------------------

import ipywidgets as widgets
from IPython.display import display

class UIChassis:
    def __init__(self, sov_tab, oracle_tab, metrics_tab):
        self.sov     = sov_tab
        self.oracle  = oracle_tab
        self.metrics = metrics_tab
        self.tab     = None

    def assemble(self):
        self.tab = widgets.Tab()
        self.tab.children = [
            self.sov.get_render_box(),
            self.oracle.get_render_box(),
            self.metrics.get_render_box()
        ]
        self.tab.set_title(0, '🎮 Command Center')
        self.tab.set_title(1, '👁️ Oracle')
        self.tab.set_title(2, '📊 Metrics')

    def display(self):
        if self.tab: display(self.tab)

    def render(self):
        pass

# ------------------------------------------
# IPO CHECK:
# Input:   CommandCenter, UIOracleTab, UIMetrics instances
# Process: Wrap in Tab widget, assign titles
# Output:  Rendered tabbed UI
# ------------------------------------------


In [ ]:
# ==========================================
# NOAH'S ARK v1.9.0: MASTER ASSEMBLY — Sprint 2
# ==========================================
# Sprint 2 — Chapter 2.3 — Multi-Character Scene Reliability
# Build order: T118 → T000 → T100 → T107 → T102 → T115 → T103
#            → T104 → T108 → T109 → T106 → T114 → T111 → T112
#            → T113 → T110 → T117 → Wire → Observe → Launch
# ------------------------------------------

# ── T118 + T000: Boot ─────────────────────────────────────────────────────────
env_loader = EnvLoader()
provider, model_name, genai_client = boot_sequence(env_loader)
api_key = env_loader.get("GEMINI_API_KEY") or env_loader.get("AGISK_Default")
initialize_env(provider)

# ── Core State + Shield + Provider ────────────────────────────────────────────
state    = StateEngine()
shield   = QuotaShield()
gateway  = GeminiProvider(api_key, model_name, client=genai_client)
vault    = IdentityVault(state)

# ── Processing Tiles ──────────────────────────────────────────────────────────
sentinel  = TokenSentinel()
archiver  = LogArchiver()
slider    = ContextSlider()
maya      = MayaMetaObserver(gateway, shield)

# ── Registry Bootstrap ────────────────────────────────────────────────────────
registry  = {}
brain     = CharacterBrain(gateway, shield, sentinel, registry)
conductor = SimConductor(registry)

registry.update({
    'state':     state,
    'shield':    shield,
    'gateway':   gateway,
    'vault':     vault,
    'sentinel':  sentinel,
    'archiver':  archiver,
    'slider':    slider,
    'brain':     brain,
    'maya':      maya,
    'conductor': conductor,
})

# ── UI Components ─────────────────────────────────────────────────────────────
ui_cmd    = CommandCenter(registry)
ui_oracle = UIOracleTab()
ui_met    = UIMetrics()

registry['ui_cmd']    = ui_cmd
registry['conductor'] = conductor

# ── Link UI to Logic ──────────────────────────────────────────────────────────
oracle_logic        = OracleLogic(registry)
registry['oracle']  = oracle_logic
ui_oracle.r         = registry
ui_met.sentinel_handle = sentinel

# ── Final Assembly (T117) ─────────────────────────────────────────────────────
chassis = UIChassis(ui_cmd, ui_oracle, ui_met)
chassis.assemble()

# ── Global Tab Observer ───────────────────────────────────────────────────────
def system_observer(change):
    if change['new'] == 2:
        ui_met.refresh_data()
    elif change['new'] == 0:
        if hasattr(ui_cmd, '_refresh_cockpit'):
            ui_cmd._refresh_cockpit()

chassis.tab.observe(system_observer, names='selected_index')

# ── Wire Console Logging ──────────────────────────────────────────────────────
ui_cmd.wire_logging()

# 🚀 LAUNCH
chassis.display()
